## Tutorial 
# Compare NEON and EMIT data for SOAP site
## First of two notebooks
### Authors: Randi Neff, Hannah Rieder and Bridget Hass
#### last updated: 7/31/25

This tutorial is intended for Earth Science data professionals. Additional details and an overall summary of this project is available on the [Earth Lab Blog](https://earthlab.colorado.edu/earth-data-analytics-professional-graduate-certificate/earth-data-analytics-certificate-cohorts). In this tutorial, we will learn how to evaluate forest health using a calculation of the Canopy Water Content (CWC) from individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. The hyperspectral data for the CWC calculation comes from the National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface directional reflectance - mosaic data product and the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product. 

## The objectives of this tutorial (divided between two notebooks) are to:
* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare CWC calculations between burned and unburned areas

DATA
The data provided with this tutorial were derived from existing code at:
* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for Creek fire boundary and NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* [Land Processes Distributed Active Archive Center (LP DAAC)](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#cwc-of-a-single-point).

Additional data will be downloaded programmatically within this tutorial.

# 1.0 Setup and Functions

## Tutorial Outline for Notebook 1 - NEON and EMIT Data 
* You will need a NEON user account, but will be provided with shapefiles for burned/unburned tile boundaries 
* NEON tile boundaries will be used to crop EMIT data to the same region of interest (ROI)
* You will need a NASA Earthdata account and functions found in the script folder

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison
* Open NEON and EMIT Reflectance Data
* Calculate Canopy Water Content (CWC)
* Compare CWC Datasets

## Notes on functions that are used in both notebooks and found in the scripts folder
* The two datasets (NEON & EMIT) are very large and in different formats so the aop_h5refl2xarray function does conversions to make them compatible
* The calc_ewt function was originally developed for EMIT data and some metadata is hardcoded so preserved while maintaining functionality
* The data_download_tracker allows the two datasets to be stored and found in different directories
* The surfrfl_hvplot_image assists with visualizing both datasets

Additional considerations are discussed in the [README file](https://github.com/NEONScience/AOP-EMIT/blob/main/README.md)

In [2]:
# Import required libraries - revise based on what we actually include with tutorial
import os, sys # management of files and directories
import requests # downloading data from online sources
import earthaccess # accessing NASA Earth data
import folium # visualizing geospatial data
import warnings # customizing how Python handles non-critical issues
import csv # working with tabular data
import pandas as pd # data manipulation with dataframes and 1D arrays  
import geopandas as gpd # add geometry to panda dataframes
import rasterio as rio # raster library
import xarray as xr # working with multi-dimensional arrays
import holoviews as hv # interactive visualizations
import hvplot.xarray # graphing
import netCDF4 as nc # read and write NetCDF files

from zipfile import ZipFile # handling data that comes in zipped formats
from branca.element import Figure # structuring the HTML output
from IPython.display import display # working in Jupyter notebooks
from rasterio.plot import show, show_hist # Generates and displays a histogram 
# of the raster data

# This will ignore some warnings caused by holoviews
warnings.simplefilter('ignore') 

# Include test_functions.py
from modules.emit_tools import emit_xarray #open EMIT datasets into xarray.Dataset
from modules.test_functions import surfrfl_hvplot_image

ModuleNotFoundError: No module named 'modules'

If not already installed, install the neonutilities and python-dotenv packages 
using pip as follows:
!pip install neonutilities
!pip install python-dotenv

In [ ]:
import neonutilities as nu

Login to your NASA Earthdata account and 
create a .netrc file using the login function from the earthaccess library. 
If you do not have an Earthdata Account, you can create one here.

In [ ]:
earthaccess.login(persist=True)

For this notebook:
* NEON & EMIT co-located data info here
* we will download the files necessary using earthaccess.
* You can also access the data in place or stream it, but this can slow due to the file sizes.
* Provide a URL for an EMIT L2A Reflectance granule.

In [ ]:
url = 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230731T205320_2321214_004/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc'

Get an HTTPS Session using your earthdata login, set a local path to save the file, and download the granule asset - This may take a while, the reflectance file is approximately 1.8 GB.

In [ ]:
# Get requests https Session using Earthdata Login Info
fs = earthaccess.get_requests_https_session()
# Retrieve granule asset ID from URL (to maintain existing naming convention)
granule_asset_id = url.split('/')[-1]
# Define Local Filepath
fp = f'./data/REFL/{granule_asset_id}'
# Download the Granule Asset if it doesn't exist
if not os.path.isfile(fp):
    with fs.get(url,stream=True) as src:
        with open(fp,'wb') as dst:
            for chunk in src.iter_content(chunk_size=64*1024*1024):
                dst.write(chunk)

# 1.2 NEON Region of Interest 
* Create a NEON API token following [this tutorial](https://www.neonscience.org/resources/learning-hub/tutorials/neon-api-tokens-tutorial).
* Load a shapefile of the NEON site boundaries and SOAP reflectance data.


In [ ]:
# set up neon token
# NEON_TOKEN = "PASTE YOUR TOKEN HERE"

In [ ]:
# Use NEON token for data download
NEON_TOKEN = "eyJ0eXAiOiJKV1QiLCJhbGciOiJFUzI1NiJ9.eyJhdWQiOiJodHRwczovL2RhdGEu"
"bmVvbnNjaWVuY2Uub3JnL2FwaS92MC8iLCJzdWIiOiJyX25lZmZAc291dGh3ZXN0ZXJuY2MuZWR1Ii"
"wic2NvcGUiOiJyYXRlOnB1YmxpYyIsImlzcyI6Imh0dHBzOi8vZGF0YS5uZW9uc2NpZW5jZS5vcmcv"
"IiwiZXhwIjoxOTExMzU2NDU5LCJpYXQiOjE3NTM2NzY0NTksImVtYWlsIjoicl9uZWZmQHNvdXRod2"
"VzdGVybmNjLmVkdSJ9.W8p1lLMUIaK8aynlbx3ZJbtSNpFWdUsV2dSmTHCrPv8c4UhNCCEb-t7y3HT"
"ln61ZWyxsH8cDU_tKFoDSvFkuJg"

In [ ]:
# Use token to download AOP data
# nu.by_tile_aop(dpid='DP3.30015.001',
#                site='SOAP',
#                year='2023',
#                easting=[298000, 298000],
#                northing=[4100000, 4101000],
#                include_provisional=True,
#                token=os.environ.get("NEON_TOKEN"),
#                savepath='./data/NEON') - update with final path

In [ ]:
# function to download data stored on the internet in a public url to a local file 
def download_url(url,download_dir):
    if not os.path.isdir(download_dir):
        os.makedirs(download_dir)
    filename = url.split('/')[-1]
    r = requests.get(url, allow_redirects=True)
    file_object = open(os.path.join(download_dir,filename),'wb')
    file_object.write(r.content)

In [ ]:
# Download and Unzip the NEON Flight Boundary Shapefile 
neon_boundary_url = "https://www.neonscience.org/sites/default/files/AOP_flightBoxes_0.zip"
# Use download_url function to save the file to a directory
os.makedirs('./data', exist_ok=True)
download_url(neon_boundary_url,'./data')
# Unzip the file
with ZipFile(f"./data/{neon_boundary_url.split('/')[-1]}", 'r') as zip_ref:
    zip_ref.extractall('./data')

In [ ]:
aop_flightboxes = gpd.read_file("./data/AOP_flightBoxes/AOP_flightboxesAllSites.shp")
aop_flightboxes.head()

In [ ]:
site_id = 'SOAP'
aop_flightboxes[aop_flightboxes.siteID == site_id]

# 1.3 EMIT Data

EMIT L2A Reflectance Data are distributed in a non-orthocorrected spatially raw NetCDF4 (.nc) format consisting of the data and its associated metadata. To work with this data, we will use the emit_xarray function from the emit_tools.py module included in the repository.

In [ ]:
ds_nc = nc.Dataset(fp)
ds_nc

In [ ]:
ds_nc['location']

# 1.4 Crop EMIT Data

To make the rest of this code run quicker and to make the CWC calculation less intensive, crop the EMIT granule to the SOAP flight boxes now. Taken from [Hannah's 07 notebook](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/07_hr_cwc_emit.ipynb)

In [ ]:
#open a shapefile of the ROI
aop_flightboxes = gpd.read_file("../../../data/shapefiles/aop_flightboxes/AOP_flightboxesAllSites.shp")
soap_polygon = aop_flightboxes[aop_flightboxes.siteID == 'SOAP']
shape = soap_polygon
shape

In [ ]:
#define EMIT file path
emit_fp = ("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl"
           "/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc")

In [ ]:
#learn about emit_xarray function
help(emit_xarray)

In [ ]:
#open emit_fp
emit_ds = emit_xarray(
    #filepath
    emit_fp,
    #orthorectify the dataset
    ortho=True
).load()

#check dataset
emit_ds

In [ ]:
#crop emit_ds to SOAP flightboxes
emit_crop_ds = emit_ds.rio.clip(
    #crop to SOAP polygon geometry
    shape.geometry.values,
    #crop to SOAP polygon CRS
    shape.crs,
    #include all pixels touched by polygon
    all_touched=True)

In [ ]:
#export emit_crop_ds and save to filepath we can use in calc_ewt fxn
emit_crop_ds.to_netcdf("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP.nc")

#define filepath
emit_crop_fp = ("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP.nc")

#check filepath
emit_crop_fp

In [ ]:
# open emit_crop_fp using code from line 37 of ewt_calc.py
# we did this to see if this helps the final fxn run faster
# we also did this to open the cropped dataset in the same way that the 
# calc_ewt fxn does to see if that helps reduce errors.
emit_crop_ds = xr.open_dataset(emit_crop_fp, decode_coords="all")

# check dataset
emit_crop_ds

In [ ]:
# check emit_crop_ds.reflectance after loading in w/ ewt_calc.py code
emit_crop_ds.reflectance

In [ ]:
# check emit_crop_ds.reflectance after loading in w/ ewt_calc.py code.
# wanting to check NaN values and reflectance values in general
# to make sure they're NaN values and not -3000000 from cropping and exporting above
emit_crop_ds.reflectance.plot.hist()

In [ ]:
# view surface reflectance of cropped area for wavelength closest to 850
emit_crop_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# open burned tile of interest shapefile

# define filepath for burned tiles
burned_tile_shp_fp = ('..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001'
                   '\\neon-aop-products\\2023\\FullSite\\D17'
                   '\\2023_SOAP_7\\Metadata\\DiscreteLidar'
                   '\\TileBoundary\\shps'
                   '\\NEON_D17_SOAP_DPQA_298000_4100000_boundary.shp')

# write burned tile boundary filepath to geodataframe
burned_tile_gdf = gpd.read_file(burned_tile_shp_fp)

# check geodataframe
burned_tile_gdf

In [ ]:
# open unburned tile of interest shapefile

# define filepath for burned tiles
unburned_tile_shp_fp = ('..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001'
                   '\\neon-aop-products\\2023\\FullSite\\D17'
                   '\\2023_SOAP_7\\Metadata\\DiscreteLidar'
                   '\\TileBoundary\\shps'
                   '\\NEON_D17_SOAP_DPQA_298000_4101000_boundary.shp')

# write burned tile boundary filepath to geodataframe
unburned_tile_gdf = gpd.read_file(burned_tile_shp_fp)

# check geodataframe
unburned_tile_gdf

In [ ]:
# crop EMIT granule to burned tile of interest
emit_burn_ds = emit_crop_ds.rio.clip(
    #crop to burned tile polygon geometry
    burned_tile_gdf.geometry.values,
    #crop to burned tile polygon CRS
    burned_tile_gdf.crs,
    #include all pixels touched by polygon
    all_touched=True)
# check emit_burn_ds
emit_burn_ds

In [ ]:
# crop EMIT granule to burned tile of interest
emit_unburn_ds = emit_crop_ds.rio.clip(
    #crop to burned tile polygon geometry
    unburned_tile_gdf.geometry.values,
    #crop to burned tile polygon CRS
    unburned_tile_gdf.crs,
    #include all pixels touched by polygon
    all_touched=True)
# check emit_burn_ds
emit_unburn_ds

In [ ]:
# view surface reflectance of burned area for wavelength closest to 850
emit_burn_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# view surface reflectance of burned area for wavelength closest to 850
emit_unburn_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# export emit_burn_ds and save to filepath we can use in calc_ewt fxn
emit_burn_ds.to_netcdf("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")

# define filepath
emit_burn_fp = ("../../../data"
           "/SOAP"
           "/EMIT"
           "/L2Arefl/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")

# check filepath
emit_burn_fp

In [ ]:
# Add unburned EMIT

# 1.5 NEON Tiles

In [ ]:
# Repeat for NEON both tiles

# 1.6 Visualize Data

In [ ]:
# plot burned dataset to check it was loaded back in correctly
surfrfl_hvplot_image(
    emit_burn_ds.sel(
        wavelengths=850,
        # use nearest valid index value
        method='nearest'),
    plottitle='SOAP Burned Tile EMIT Surface Reflectance, 850.1 nm')